# Previsão Glicêmica (AZT1D) — Escada de Representações de Insulina e Carboidratos

Este notebook segue a orientação de `Previsao_Glicemica_AZT1D_Metodos_e_Plano.md` como referência
principal do projeto. Ele implementa o que o documento chama de **Roteiro enxuto** restrito à
**trilha A (modelos tabulares)**, que o próprio documento diz que "deve existir mesmo que B ou C
sejam escolhidas como contribuição principal" (Seção 6).

## O que este notebook implementa, e de qual seção do documento vem

| Elemento | Seção do documento |
|---|---|
| Duas tabelas canônicas (observações de CGM + eventos de insulina/carboidratos) | 9.2 |
| Máscara de cobertura do CGM (leitura real vs. interpolada) | 4.3 |
| Representação por **impulsos brutos** (dose/carboidrato no próprio bin de 5 min) | 5.2 |
| Representação por **recências** (somas em faixas de atraso: 0–30, 30–60, 60–120, 120–240 min) | 5.2 |
| Representação por **kernel causal de efeito residual** (modelo de 2 compartimentos iguais) | 5.3–5.4 |
| Cálculo dos estados de forma causal sobre a série completa, sem reiniciar a cada janela | 5.7 |
| Regra de não vazamento: evento futuro não entra na entrada; alvo não escapa da partição | 4.4, 10.2 |
| Baseline de persistência (P0) e escada de modelos A0→A3 | 9.3, 10.6 |
| Horizontes de 30, 60 e 120 min, um regressor por horizonte | 7.2, 9.6 |
| Métricas: MAE/RMSE globais, RMSE macro por paciente, *skill* vs. persistência | 10.4 |
| Estratificação por faixa glicêmica | 10.5 |

## O que fica **fora** do escopo deste notebook (e por quê)

O documento descreve uma trilha muito mais ampla (Seções 2–3 e 9–11): LSTM com estados residuais
(trilha B), modelo acionado pelo registro de refeição com dois ramos (trilha C1), GRU-D para
irregularidade, personalização por paciente, perdas sensíveis a faixa glicêmica, intervalos
preditivos, avaliação leave-subject-out com 25 participantes, análise de sensibilidade ao
Subject 14, e uso do campo Basal (deixado de fora enquanto sua unidade/escala não for esclarecida —
Seção 4.2/9.4).

Isso é consciente, não uma omissão por esquecimento: a Seção 11 do próprio documento recomenda
começar pelo **roteiro enxuto** (comparação controlada de persistência, modelos tabulares e,
depois, LSTM) antes de partir para arquiteturas mais caras. Este notebook cobre a parte tabular
desse roteiro. A explicação completa de cada decisão vem depois, na conversa.


In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib

## 1. Conectar ao Google Drive e localizar o dataset

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    SEARCH_ROOT = Path("/content/drive/MyDrive")
except ImportError:
    SEARCH_ROOT = Path.cwd()

def find_dataset_dir(search_root, folder_name="CGM Records"):
    for cand in search_root.rglob(folder_name):
        if cand.is_dir() and list(cand.glob("Subject */Subject *.csv")):
            return cand
    return None

DATASET_DIR = find_dataset_dir(SEARCH_ROOT)
if DATASET_DIR is None:
    raise FileNotFoundError("Pasta 'CGM Records' não encontrada no Drive.")

print("Dataset encontrado em:", DATASET_DIR)

## 2. Configuração

`SUBJECTS` continua em 3 pacientes, como vínhamos usando — o documento recomenda eventualmente
avaliar com mais participantes e divisões *leave-subject-out* (Seção 10.1), o que fica registrado
como próximo passo na seção final.

Os parâmetros `THETA_I` e `THETA_C` seguem o exemplo numérico da Seção 5.4 do documento
(`θ_I = 60 min`); eles são **escolhas de engenharia documentadas**, não valores farmacológicos
validados clinicamente — o próprio texto é explícito sobre isso (Seção 5.4 e 5.6). O ideal, segundo
a Seção 9.6, seria uma pequena busca desses parâmetros em treino/validação; aqui usamos valores
fixos para manter o notebook simples, e isso entra como limitação declarada.


In [ ]:
STEP_MIN = 5
LAG_STEPS = 24                 # ~2h de histórico de CGM (convenção descrita na Seção 9.4)
HORIZONS_MIN = [30, 60, 120]   # alvos propostos na Seção 7.2
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15   # o restante (0.15) é teste

SUBJECTS = ["Subject 1", "Subject 2", "Subject 3"]

THETA_I = 60.0   # min — exemplo numérico da Seção 5.4 (não é valor clínico validado)
THETA_C = 45.0   # min — escolha de engenharia; carboidrato tende a agir mais rápido que insulina

RECENCY_BINS_MIN = [(0, 30), (30, 60), (60, 120), (120, 240)]   # faixas de atraso da Seção 5.2

## 3. Duas tabelas canônicas por paciente (Seção 9.2)

Em vez de misturar tudo em um único DataFrame bruto, separamos conceitualmente:

- **Observações**: a leitura de CGM reamostrada a cada 5 min, com uma **máscara de cobertura**
  (`mask_cgm`) indicando se aquele ponto veio de uma leitura real do sensor ou foi interpolado —
  essa distinção é a recomendação da Seção 4.3 ("ausência de evento não é ausência de efeito").
- **Eventos**: a dose de insulina (`CorrectionDelivered + TotalBolusInsulinDelivered`) e os
  carboidratos (`CarbSize + FoodDelivered`) somados dentro de cada bin de 5 min — sem repetir o
  mesmo evento em mais de um bin (Seção 4.2).

O campo `Basal` é deixado de fora: o documento explica que sua unidade/escala continua ambígua na
auditoria do dataset, e recomenda não usá-lo até isso ser esclarecido (Seção 4.2, 9.4, 5.5).


In [ ]:
import pandas as pd
import numpy as np

def load_subject_tables(subject_name):
    path = DATASET_DIR / subject_name / f"{subject_name}.csv"
    df = pd.read_csv(path)
    df["EventDateTime"] = pd.to_datetime(df["EventDateTime"])
    df = df.sort_values("EventDateTime").set_index("EventDateTime")
    df = df[~df.index.duplicated(keep="first")]

    # --- tabela de observações (CGM) ---
    raw_cgm = df["CGM"].resample(f"{STEP_MIN}min").mean()
    mask_cgm = raw_cgm.notna().astype(float)       # 1 = leitura real, 0 = lacuna
    cgm = raw_cgm.interpolate(limit=6)              # só preenche lacunas curtas (até 30 min)

    # --- tabela de eventos (insulina / carboidratos) ---
    events = df.resample(f"{STEP_MIN}min").agg({
        "CorrectionDelivered": "sum",
        "TotalBolusInsulinDelivered": "sum",
        "CarbSize": "sum",
        "FoodDelivered": "sum",
    }).fillna(0.0)

    out = pd.DataFrame(index=cgm.index)
    out["cgm"] = cgm
    out["mask_cgm"] = mask_cgm.reindex(out.index).fillna(0.0)
    out["insulina_bin"] = events["CorrectionDelivered"] + events["TotalBolusInsulinDelivered"]
    out["carbo_bin"] = events["CarbSize"] + events["FoodDelivered"]

    out = out.dropna(subset=["cgm"])   # remove lacunas longas que não puderam ser interpoladas
    return out

## 4. Atributos causais: recências e kernel de efeito residual (Seção 5.2–5.4)

### 4.1 Recências

Em vez de olhar só o valor bruto do bin atual (que é zero na maior parte do tempo), somamos
insulina e carboidratos em **faixas de atraso**: últimos 0–30 min, 30–60, 60–120 e 120–240 min.
Essas faixas são, como o documento deixa claro, "uma proposta de engenharia para ablação, não
durações fisiológicas estabelecidas" (Seção 5.2).


In [ ]:
def add_recency_features(df):
    for lo, hi in RECENCY_BINS_MIN:
        steps_hi, steps_lo = hi // STEP_MIN, lo // STEP_MIN

        roll_hi_i = df["insulina_bin"].rolling(steps_hi, min_periods=1).sum()
        roll_lo_i = df["insulina_bin"].rolling(steps_lo, min_periods=1).sum() if steps_lo > 0 else 0
        df[f"insulina_rec_{lo}_{hi}"] = roll_hi_i - roll_lo_i

        roll_hi_c = df["carbo_bin"].rolling(steps_hi, min_periods=1).sum()
        roll_lo_c = df["carbo_bin"].rolling(steps_lo, min_periods=1).sum() if steps_lo > 0 else 0
        df[f"carbo_rec_{lo}_{hi}"] = roll_hi_c - roll_lo_c
    return df

### 4.2 Kernel causal de 2 compartimentos (efeito residual)

A Seção 5.4 do documento dá a solução fechada para um depósito único $D$ atravessando dois
compartimentos iguais com constante de tempo $\theta$:

$$R_\theta(a)=e^{-a/\theta}\left(1+\frac{a}{\theta}\right), \qquad k_\theta(a)=\frac{a}{\theta^2}e^{-a/\theta}$$

Em vez de recalcular essa soma para todo o histórico a cada passo (custoso), usamos a forma
recursiva exata desse mesmo sistema — o resultado matemático é idêntico, mas o cálculo fica O(n) em
vez de O(n²), e os estados são computados **de forma causal sobre a série completa antes de
recortar qualquer janela**, como pede a Seção 5.7 ("não reiniciar os estados a zero em cada
janela").


In [ ]:
def two_compartment_states(doses, dt_min, theta_min):
    """Estados causais exatos (R = quantidade remanescente, F = fluxo de saída) do
    modelo de 2 compartimentos iguais da Seção 5.3-5.4, via recursão O(n)."""
    decay = np.exp(-dt_min / theta_min)
    k = dt_min / theta_min
    s1 = s2 = 0.0
    R = np.empty(len(doses))
    F = np.empty(len(doses))
    for i, d in enumerate(doses):
        s2 = (s2 + k * s1) * decay
        s1 = s1 * decay + d
        R[i] = s1 + s2
        F[i] = s2 / theta_min
    return R, F


def add_kernel_features(df):
    R_i, F_i = two_compartment_states(df["insulina_bin"].values, STEP_MIN, THETA_I)
    R_c, F_c = two_compartment_states(df["carbo_bin"].values, STEP_MIN, THETA_C)
    df["insulina_remanescente"], df["insulina_fluxo"] = R_i, F_i
    df["carbo_remanescente"], df["carbo_fluxo"] = R_c, F_c
    return df

### 4.3 Conferindo a recursão contra a fórmula fechada do documento

Antes de usar isso em qualquer modelo, vale conferir que a recursão bate com o exemplo numérico
que o próprio documento dá: uma dose isolada de $D=4$ U com $\theta=60$ min deveria deixar
$I_{\mathrm{rem}}\approx2{,}94$ U e um fluxo $F_I\approx0{,}0245$ U/min após 60 minutos
(Seção 5.4).


In [ ]:
doses_teste = np.zeros(200)
doses_teste[0] = 4.0  # dose única de 4U no instante 0

R_teste, F_teste = two_compartment_states(doses_teste, STEP_MIN, 60.0)
passo_60min = 60 // STEP_MIN  # 60 min após a dose

print(f"R em t=60min (recursão):        {R_teste[passo_60min]:.4f} U")
print(f"R em t=60min (fórmula do doc.): {np.exp(-1)*(1+1)*4.0:.4f} U   (esperado ≈ 2.94 U)")
print(f"F em t=60min (recursão):        {F_teste[passo_60min]:.4f} U/min")
print(f"F em t=60min (fórmula do doc.): {(60/60**2)*np.exp(-1)*4.0:.4f} U/min   (esperado ≈ 0.0245 U/min)")

## 5. Montar os atributos causais completos de cada paciente

In [ ]:
def build_subject_features(subject_name):
    df = load_subject_tables(subject_name)
    minutos_dia = df.index.hour * 60 + df.index.minute
    df["hora_sin"] = np.sin(2 * np.pi * minutos_dia / 1440)
    df["hora_cos"] = np.cos(2 * np.pi * minutos_dia / 1440)
    df = add_recency_features(df)
    df = add_kernel_features(df)
    return df

subject_frames = {s: build_subject_features(s) for s in SUBJECTS}
for s, df in subject_frames.items():
    print(f"{s}: {len(df)} pontos, cobertura de CGM real = {df['mask_cgm'].mean()*100:.1f}%")

## 6. Escada de representações e baseline de persistência (Seção 9.3, 10.6)

Quatro conjuntos de atributos, cada um contendo o anterior + um tipo novo de informação — assim dá
pra isolar exatamente o que cada representação acrescenta, como pede a tabela de ablações da
Seção 10.6:

| Modelo | O que tem a mais que o anterior |
|---|---|
| `A0_cgm_only` | (nada — só histórico de CGM, variação, hora do dia, máscara) |
| `A1_impulsos` | + dose/carboidrato bruto do bin atual |
| `A2_recencias` | + somas por faixa de atraso (0–30, 30–60, 60–120, 120–240 min) |
| `A3_kernels` | + estados do kernel causal (quantidade remanescente e fluxo) |

`P0_persistencia` (Seção 9.3) é o baseline mais simples — "a glicose em `h` minutos será igual à
glicose agora" — contra o qual todo o resto é comparado.


In [ ]:
FEATURE_SETS = {
    "A0_cgm_only": [],
    "A1_impulsos": ["insulina_bin", "carbo_bin"],
    "A2_recencias": ["insulina_bin", "carbo_bin"]
        + [f"insulina_rec_{lo}_{hi}" for lo, hi in RECENCY_BINS_MIN]
        + [f"carbo_rec_{lo}_{hi}" for lo, hi in RECENCY_BINS_MIN],
    "A3_kernels": ["insulina_bin", "carbo_bin"]
        + [f"insulina_rec_{lo}_{hi}" for lo, hi in RECENCY_BINS_MIN]
        + [f"carbo_rec_{lo}_{hi}" for lo, hi in RECENCY_BINS_MIN]
        + ["insulina_remanescente", "insulina_fluxo", "carbo_remanescente", "carbo_fluxo"],
}

## 7. Amostras sem vazamento temporal (Seção 4.4, 10.2)

Duas regras são aplicadas ao construir cada amostra $(\mathbf x_{t_0}, G_{t_0+h})$:

1. **Nenhum evento posterior a $t_0$ entra na entrada** — como todos os atributos (recências,
   kernels) são computados causalmente, isso já vem garantido pela forma como foram construídos.
2. **O alvo não pode escapar da partição** — se $t_0$ está no treino mas $t_0+h$ cairia dentro do
   período reservado para teste, essa amostra é descartada. Isso segue a "política conservadora"
   descrita na Seção 10.2: o intervalo completo de entrada e alvo precisa ficar dentro da mesma
   partição.


In [ ]:
def split_bounds(n, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC):
    a, b = int(n * train_frac), int(n * (train_frac + val_frac))
    return {"train": (0, a), "val": (a, b), "test": (b, n)}


def build_samples(df, extra_cols, horizon_steps, lo, hi):
    cgm = df["cgm"].values
    mask = df["mask_cgm"].values
    hora_sin = df["hora_sin"].values
    hora_cos = df["hora_cos"].values
    extra = df[extra_cols].values if extra_cols else None

    X, y, idx = [], [], []
    start = max(lo, LAG_STEPS)
    end = min(hi, len(df) - horizon_steps + 1)
    for i in range(start, end):
        target_pos = i - 1 + horizon_steps
        if target_pos >= hi:        # regra: alvo não escapa da partição
            continue
        row = list(cgm[i - LAG_STEPS:i])                        # lags de CGM (2h)
        row.append(cgm[i - 1] - cgm[i - 2] if i >= 2 else 0.0)   # delta_cgm
        row.append(hora_sin[i - 1]); row.append(hora_cos[i - 1])
        row.append(mask[i - 1])                                  # máscara de cobertura
        if extra is not None:
            row.extend(extra[i - 1])
        X.append(row); y.append(cgm[target_pos]); idx.append(i - 1)
    return np.array(X), np.array(y), idx

## 8. Treinar e avaliar: persistência + escada A0–A3, para 30/60/120 min

Um regressor **Ridge** por horizonte (Seção 9.6 recomenda um regressor por horizonte, evitando
decodificador recursivo), com o *scaler* ajustado só no treino. As métricas seguem a Seção 10.4:

- **MAE/RMSE globais** (todas as amostras de teste juntas);
- **RMSE macro** (erro calculado por paciente e depois tirada a média — evita que o paciente com
  mais linhas domine a conclusão);
- ***Skill* vs. persistência** = $1 - \mathrm{RMSE_{modelo}} / \mathrm{RMSE_{persistência}}$ —
  positivo indica ganho real sobre o baseline mais simples.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

results = []

for h_min in HORIZONS_MIN:
    h_steps = h_min // STEP_MIN

    # --- baseline de persistência no conjunto de teste de cada paciente ---
    per_subject_test = {}
    for s in SUBJECTS:
        df = subject_frames[s]
        bounds = split_bounds(len(df))
        Xte0, yte, idx_te = build_samples(df, [], h_steps, *bounds["test"])
        per_subject_test[s] = {"y": yte, "persist": df["cgm"].values[idx_te]}

    y_all = np.concatenate([v["y"] for v in per_subject_test.values()])
    p_all = np.concatenate([v["persist"] for v in per_subject_test.values()])
    rmse_persist_global = np.sqrt(mean_squared_error(y_all, p_all))
    mae_persist_global = mean_absolute_error(y_all, p_all)
    rmse_persist_macro = np.mean(
        [np.sqrt(mean_squared_error(v["y"], v["persist"])) for v in per_subject_test.values()]
    )
    results.append({"horizonte_min": h_min, "modelo": "P0_persistencia",
                     "MAE_global": mae_persist_global, "RMSE_global": rmse_persist_global,
                     "RMSE_macro": rmse_persist_macro, "skill_RMSE": 0.0})

    # --- escada de modelos A0..A3 ---
    for fname, extra_cols in FEATURE_SETS.items():
        Xtr_l, ytr_l, Xte_l, yte_l, subj_l = [], [], [], [], []
        for s in SUBJECTS:
            df = subject_frames[s]
            bounds = split_bounds(len(df))
            Xtr, ytr, _ = build_samples(df, extra_cols, h_steps, *bounds["train"])
            Xte, yte, _ = build_samples(df, extra_cols, h_steps, *bounds["test"])
            Xtr_l.append(Xtr); ytr_l.append(ytr)
            Xte_l.append(Xte); yte_l.append(yte)
            subj_l += [s] * len(Xte)

        X_train, y_train = np.concatenate(Xtr_l), np.concatenate(ytr_l)
        X_test, y_test = np.concatenate(Xte_l), np.concatenate(yte_l)
        subj_arr = np.array(subj_l)

        sx = StandardScaler(); Xtr_s = sx.fit_transform(X_train); Xte_s = sx.transform(X_test)
        sy = StandardScaler(); ytr_s = sy.fit_transform(y_train.reshape(-1, 1)).ravel()

        model = Ridge(alpha=1.0)
        model.fit(Xtr_s, ytr_s)
        pred = sy.inverse_transform(model.predict(Xte_s).reshape(-1, 1)).ravel()

        mae_g = mean_absolute_error(y_test, pred)
        rmse_g = np.sqrt(mean_squared_error(y_test, pred))
        rmse_macro = np.mean([
            np.sqrt(mean_squared_error(y_test[subj_arr == s], pred[subj_arr == s])) for s in SUBJECTS
        ])
        skill = 1 - rmse_g / rmse_persist_global

        results.append({"horizonte_min": h_min, "modelo": fname,
                         "MAE_global": mae_g, "RMSE_global": rmse_g,
                         "RMSE_macro": rmse_macro, "skill_RMSE": skill})

        if fname == "A3_kernels":  # guarda previsões do modelo mais completo p/ seções seguintes
            globals()[f"pred_A3_{h_min}min"] = pred
            globals()[f"ytest_A3_{h_min}min"] = y_test
            globals()[f"subj_A3_{h_min}min"] = subj_arr

resultados_df = pd.DataFrame(results)
resultados_df.round(3)

## 9. Visualizar o *skill* de cada modelo por horizonte

Um *skill* positivo e crescente da esquerda (A0) para a direita (A3) seria evidência de que cada
camada de representação (impulsos → recências → kernels) acrescenta informação útil. Um *skill*
praticamente igual entre os modelos é também um resultado válido — significa que, para estes 3
pacientes e com Regressão Ridge, a forma de representar insulina/carboidratos importou pouco diante
do histórico de CGM (ver Seção 10.6 e a discussão de limites na Seção 1.4).


In [ ]:
import matplotlib.pyplot as plt

pivot = resultados_df[resultados_df["modelo"] != "P0_persistencia"].pivot(
    index="modelo", columns="horizonte_min", values="skill_RMSE"
).loc[["A0_cgm_only", "A1_impulsos", "A2_recencias", "A3_kernels"]]

pivot.plot(kind="bar", figsize=(9, 4))
plt.ylabel("Skill vs. persistência (RMSE)")
plt.title("Ganho de cada representação sobre a persistência, por horizonte")
plt.legend(title="Horizonte (min)")
plt.tight_layout()
plt.show()

## 10. Previsto vs. real — modelo mais completo (A3), horizonte de 30 min

In [ ]:
h_plot = 30
pred, y_test, subj_arr = globals()[f"pred_A3_{h_plot}min"], globals()[f"ytest_A3_{h_plot}min"], globals()[f"subj_A3_{h_plot}min"]

mask_plot = subj_arr == SUBJECTS[0]
plt.figure(figsize=(12, 4))
plt.plot(y_test[mask_plot][:300], label="CGM real")
plt.plot(pred[mask_plot][:300], label=f"CGM previsto (+{h_plot} min, A3)", alpha=0.8)
plt.xlabel("Amostras de teste (ordem no tempo)")
plt.ylabel("Glicose (mg/dL)")
plt.title(f"Previsão a {h_plot} min — {SUBJECTS[0]} — modelo A3_kernels")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Estratificação por faixa clínica (Seção 10.5)

Como análise secundária — o documento é explícito que isso não substitui as métricas de erro
numérico da Seção 10.4 — reportamos acerto por faixa glicêmica (hipo/normal/hiperglicemia) do
modelo mais completo (A3) no horizonte de 30 min.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def categorizar(valores):
    bins = [-np.inf, 70, 180, np.inf]
    rotulos = ["Hipoglicemia", "Normal", "Hiperglicemia"]
    return pd.cut(valores, bins=bins, labels=rotulos)

y_test_30, pred_30 = globals()["ytest_A3_30min"], globals()["pred_A3_30min"]
y_cat, pred_cat = categorizar(y_test_30), categorizar(pred_30)

print(classification_report(y_cat, pred_cat, zero_division=0))
rotulos = ["Hipoglicemia", "Normal", "Hiperglicemia"]
print(pd.DataFrame(confusion_matrix(y_cat, pred_cat, labels=rotulos), index=rotulos, columns=rotulos))

## 12. Escopo e limitações (honestas, por construção)

Seguindo a própria lógica do documento (Seção 11, "Critérios para escolher o próximo passo"), o que
este notebook mostra é apenas o começo do roteiro enxuto. Ficam de fora, deliberadamente:

- **Trilha B (LSTM com estados residuais)** e **Trilha C1 (modelo acionado pela refeição, com dois
  ramos)** — a comparação principal recomendada pelo documento (Seção 11: "Escolha sugerida: B + C,
  com A como referência"). Este notebook entrega a referência A; B e C ficam como próximo passo.
- **Ajuste de `THETA_I`/`THETA_C` por validação** — usamos valores fixos documentados (Seção 9.6
  sugere uma pequena busca).
- **Avaliação *leave-subject-out*** com mais participantes (Seção 10.1) — aqui o teste é temporal
  dentro dos mesmos 3 pacientes do treino, não em pacientes nunca vistos.
- **Personalização por paciente** (Seção 8.1) e **perdas sensíveis à faixa glicêmica** (Seção 8.2).
- **Sensibilidade ao Subject 14** (Seção 4.4) — não entra porque não está entre os 3 sujeitos
  usados aqui.
- **Grade de erros de Clarke** (Seção 10.4/R28) — mencionada no documento como análise secundária;
  não incluída para manter o notebook enxuto.
- **Uso do campo `Basal`** — deixado de fora enquanto sua unidade/escala permanecer ambígua
  (Seção 4.2, 5.5, 9.4).

Essas omissões estão aqui registradas intencionalmente, para que o resultado deste notebook seja
interpretado com o escopo correto — exatamente o espírito da Seção 10 do documento.
